# exp06 — 동반구매 엣지 (Lift × α_r)

| 항목 | 값 |
|---|---|
| config | `experiments/configs/exp06_both_copurchase.yaml` |
| 결과 저장 | `experiments/results/exp06_both_copurchase/` |
| 핵심 세팅 | co_offline + co_quick 엣지, Lift값 × DiffMG α_r, temperature=0.5 |
| 목적 | Lift 가중 동반구매 엣지의 성능 기여 측정. exp01(baseline) 대비 PR-AUC 개선 여부 확인 |
| 비교 대상 | exp01(baseline), exp07(binary) |

In [ ]:
import os, sys
ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)

import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

from experiments.exp_utils import (
    run_experiment, load_experiment, compare_experiments,
    plot_alpha_heatmap, plot_training_curve, print_metrics_table, print_recommendations
)

EXP_NAME = 'exp06_both_copurchase'
CFG_PATH = 'experiments/configs/exp06_both_copurchase.yaml'
print('ROOT:', ROOT)

## 1. 학습 실행

> 이미 체크포인트가 있으면 export만 수행. 재학습 강제: `force=True`

In [ ]:
results = run_experiment(CFG_PATH, EXP_NAME)
# 재학습 강제: results = run_experiment(CFG_PATH, EXP_NAME, force=True)

## 2. 성능 지표

In [ ]:
print_metrics_table(results)

## 3. 학습 곡선 (val PR-AUC)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 4))
plot_training_curve(results.get('history', []), ax=ax)
plt.savefig(f'experiments/results/{EXP_NAME}/training_curve.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. DiffMG α_r — 관계 중요도 히트맵

> co_offline vs co_quick α_r 차이 확인. 어떤 구매 채널이 성공 예측에 더 기여하는지 해석.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
plot_alpha_heatmap(EXP_NAME, ax=ax)
plt.savefig(f'experiments/results/{EXP_NAME}/alpha_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. 순회 추천 샘플

In [ ]:
print_recommendations(results)

## 6. 3-way 비교 (baseline vs Lift×α_r vs binary×α_r)

In [ ]:
df = compare_experiments(['exp01_baseline', 'exp06_both_copurchase', 'exp07_copurchase_binary'])
display(df[['exp', 'val_pr_auc', 'val_auc_roc', 'test_pr_auc', 'test_auc_roc', 'test_f1']])